# Camera Traps in New Places - full pipeline (Colab)
**Fast path (results already in Drive):** run Cell 0, then Cell 8 (analysis) and Cell 9 (demo). Nothing is retrained.

**Full path:** Runtime -> Change runtime type -> **T4 GPU**, then run Cell 0 to 9 in order. Every step is resumable: if Colab disconnects, run Cell 0 again and continue.

After any *Restart runtime* run Cell 0 again (otherwise: `No module named 'src'`).


In [ ]:
# 0. Setup: mount Drive (results survive disconnects), find the project folder, install packages
from google.colab import drive
drive.mount('/content/drive')
import os
PROJ = '/content/drive/MyDrive/iwildcam_adapt'      # <- unzip the project here
if not os.path.isdir(PROJ + '/src') and os.path.isdir(PROJ + '/iwildcam_adapt/src'):
    PROJ = PROJ + '/iwildcam_adapt'                   # zip was unzipped one folder too deep
assert os.path.isdir(PROJ + '/src'), 'project folder not found: unzip it to MyDrive/iwildcam_adapt'
%cd {PROJ}
!pip install -q wilds timm gradio scikit-learn tqdm
CACHE = '/content/drive/MyDrive/iwildcam_cache'      # cached 224px subset (a few GB)
DATA  = '/content/wilds_data'                        # raw WILDS download (~11 GB, local disk is faster)
CKPT, RES = f'{PROJ}/ckpt', f'{PROJ}/results'
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (CPU only: training will be very slow)')

In [ ]:
# 1. Smoke test on fake data - confirms the whole pipeline runs on THIS machine before the long jobs
!bash tests/smoke.sh

In [ ]:
# 2a. Only if Cell 2 says 'HTTP Error 404' (the official CodaLab host is down): get iWildCam from a Hugging Face mirror (~12 GB, 20-40 min)
!pip install -q huggingface_hub
!python scripts/download_iwildcam_hf.py --root {DATA}

In [ ]:
# 2. Build the 224px subset cache. ONLY needed if your Drive does NOT already contain 'iwildcam_cache'
# (check: Drive -> MyDrive -> iwildcam_cache -> class_names.json). If it exists, SKIP this cell and Cell 3.
# If the official download fails with 'HTTP Error 404', run Cell 2a first.
!python -m src.build_cache --root {DATA} --cache-dir {CACHE} --train-cap-per-class 100 --per-camera-cap 400 --min-camera-images 150 --workers 4

In [ ]:
# 3. Train both backbones (ERM, last block(s) + classifier only)
!python -m src.train --backbone resnet50     --cache-dir {CACHE} --out {CKPT} --epochs 5
!python -m src.train --backbone dino_vits16  --cache-dir {CACHE} --out {CKPT} --epochs 5

In [ ]:
# 4. Baseline: seen vs unseen cameras, no adaptation
!python -m src.baseline_eval --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES}

In [ ]:
# 5. Choose TENT learning rate / steps on the unseen *validation* cameras (not the test cameras)
!python -m src.tune --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES}

In [ ]:
# 5b. (optional extension) ResNet BN-statistics shrinkage towards the source statistics, tuned on val cameras.
# --skip-tent keeps the TENT config chosen above untouched.
!python -m src.tune --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --backbones resnet50 --skip-tent --bn-priors 0 16 64 256 1024

In [ ]:
# 6. MAIN experiment: n = 16/32/64/128 unlabeled images per camera, 5 seeds, plus oracle
!python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 16 32 64 128 --seeds 0 1 2 3 4 --oracle

In [ ]:
# 7. Ablations at n=64: which layers are adapted, and share of empty frames in the adapt batch
!python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 64 --seeds 0 1 2 --scope last
for ef in [0.0, 0.25, 0.5, 0.75]:
    !python -m src.run_adaptation --cache-dir {CACHE} --ckpt-dir {CKPT} --out {RES} --methods bn tent --sizes 64 --seeds 0 1 2 --empty-frac {ef}

In [ ]:
# 8. Tables + figures -> results/summary.md and results/fig_*.png
!python -m src.analysis --results {RES} --cache-dir {CACHE} --focus-n 64
from IPython.display import Image, display
for f in ['fig_seen_unseen','fig_curve','fig_per_camera','fig_conditions','fig_empty_frac']:
    p = f'{RES}/{f}.png'
    if os.path.exists(p): display(Image(p))

In [ ]:
# 9. Gradio demo (opens a public link; the link dies when this cell stops)
# In the page choose: Backbone = dino_vits16, Adaptation = Entropy minimisation (TENT), upload 16+ photos from ONE camera.
!python -m src.demo --cache-dir {CACHE} --ckpt-dir {CKPT} --share